In [18]:
from torchvision.datasets import GTSRB
from torchvision import transforms 
import torch
import torch.nn as nn
from torch.optim import Adam
import  torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F 
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [19]:
from pathlib import Path

print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

DATA_DIR.mkdir(exist_ok=True)


C:\Users\user\deep-learning-traffic-signs\notebooks


In [20]:
basic_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=.2 , contrast=0.5),
    transforms.RandomResizedCrop((32,32),scale=(0.9,1),ratio=(1,1)),
])
train_dataset = GTSRB(
    root=str(DATA_DIR),
    split="train",
    transform=basic_transform,
    download=True
)

test_dataset = GTSRB(
    root=str(DATA_DIR),
    split="test",
    transform=basic_transform,
    download=True
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
 
)
train_N=len(train_loader.dataset)
test_N=len(test_loader.dataset)

In [23]:
class MyConvBlock( nn.Module ):
    def __init__(self,in_ch,out_ch,dropout_p):
        kernel_size=3
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size,stride=1,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2,stride=2)
        )
    def forward(self,x):
        return self.model(x)
flattened_img_size=128*4*4
N_CLASSES=43
IMG_CHS=3
IMG_WIDH=32
IMG_LENGHT=32
base_model= nn.Sequential(
    MyConvBlock(IMG_CHS,32,0), #(32,16,16)
    MyConvBlock(32,64,0.2),#(64,8,8)
    MyConvBlock(64,128,0),#(128,4,4)
    nn.Flatten(),
    nn.Linear(flattened_img_size,512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512,N_CLASSES)
)
loss_function=nn.CrossEntropyLoss()
optimizer=Adam(base_model.parameters())
torch._dynamo.config.suppress_errors = True
device=torch.device("cpu")
model=torch.compile(base_model.to(device))
model

OptimizedModule(
  (_orig_mod): Sequential(
    (0): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e

In [24]:
def get_batch_accuracy(output,y,N):
    pred=output.argmax(dim=1,keepdim=True)
    correct=pred.eq(y.view_as(pred)).sum().item()
    return correct/N


In [25]:
def train():
    loss=0
    accuracy=0
    model.train()
    for x,y in train_loader:
        output=model(x)
        optimizer.zero_grad()
        batch_loss=loss_function(output,y)
        batch_loss.backward()
        optimizer.step()
        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output,y,train_N)
    print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))
    

In [26]:
def validate():
    loss=0
    accuracy=0
    model.eval()
    with torch.no_grad():
        for x,y in test_loader:
            output=model(x)
            loss+=loss_function(output,y).item()
            accuracy += get_batch_accuracy(output,y,test_N)
        print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [27]:
epochs=15
for epoch in range (epochs):
    print ('epoch:{}'.format(epoch))   
    train()
    validate()

epoch:0


W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] WON'T CONVERT inner C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\external_utils.py line 67 
W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] due to: 
W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] Traceback (most recent call last):
W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]   File "C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2333, in __call__
W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     result = self._inner_convert(
W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]         frame, cache_entry, hooks, frame_state, skip=skip + 1
W0921 20:40:00.256000 23732 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     )
W0921 20:40:00.2

KeyboardInterrupt: 